## Overview

In this document, I use spaCy and Transformers to perform an analysis on the text of the novel The Great Gatsby.

Herein I:
1. Select a text from [Project Gutenberg](https://www.gutenberg.org/ebooks/bookshelves/search/?query=fiction%7Cadventure%7Cfantasy%7Chumor%7Chorror%7Cwestern)
2. Extract select named entities from the text.
3. Collect adjectival descriptors for the main character.
4. Perform an analysis of bigrams in the text.
5. Show how often certain characters are referenced together.

In [2]:
import re
import pandas as pd
import requests
import spacy

In [3]:
# load SpaCy language module
nlp = spacy.load("en_core_web_sm")

In [4]:
# Project Gutenberg Download Function
def download_text(url: str) -> str:
    """
    Downloads a text from the given URL. Remove header and footer text used in
    Project Gutenberg files, and collapse sequences of whitespace.
    """
    text = requests.get(url).text

    footer_index = text.rfind("*** END OF THE PROJECT GUTENBERG EBOOK ")
    text = text[:footer_index]

    header_pattern = r".*?\*\*\* START OF THE PROJECT GUTENBERG EBOOK .*? \*\*\*\s*"
    text = re.sub(header_pattern, "", text, count=1, flags=re.DOTALL)

    text = re.sub(r"\s+", " ", text)

    return text

In [5]:
# Great Gatsby url on project gutenberg
url = "https://www.gutenberg.org/ebooks/64317.txt.utf-8"

In [6]:
text = download_text(url)

In [7]:
# create SpaCy doc object
doc = nlp(text)

## Extracting Select Named Entities from Great Gatsby

Named Entity Recognition (NER) is applied to the `doc` document object created in the previous step to extract proper nouns of the following types:

- `PERSON` (characters)
- `ORG` (organizations)
- `GPE` geopolitical entities (locations, countries, cities)

These three categories capture the primary who, where, and institutional dimensions of narrative text, which form the basis for the character, and setting analyses that follow.

The results are loaded to a DataFrame with two columns:
- `entity` which contains the surface form of the named entity as it appears in the text
- `label` which contains the type of the named entity (e.g. PERSON) assigned by spacy model

In [8]:
labels = []
entities = []
target = ["PERSON", "GPE", "ORG"]
for ent in doc.ents:
  if ent.label_ in target:
    entities.append(ent.text)
    labels.append(ent.label_)

entities_labels = pd.DataFrame(
    {
        "entity": entities,
        "label": labels
    }
)
entities_labels

,entity,label
0,Thomas Parke,PERSON
1,n’t,GPE
2,Gatsby,PERSON
3,Gatsby,PERSON
4,Gatsby,PERSON
...,...,...
1405,New York,GPE
1406,Gatsby,PERSON
1407,Gatsby,PERSON
1408,Daisy,PERSON


### Value Counts of the top 20 Characters in The Great Gatsby


### (Approach 1) Just extracting NER instances using SpaCy

In [9]:
# Display the Value Counts of the top 20 characters in novel (only PERSON entities)
person_entities = entities_labels.loc[entities_labels["label"] =="PERSON",["entity"]]
person_entities.value_counts().head(20)

,count
entity,
Gatsby,239
Daisy,166
Tom,161
Wilson,35
Jordan,29
Baker,28
Wolfshiem,17
Tom Buchanan,16
McKee,15


### Analysis Approach 1: Just extracting NER instances using SpaCy

This character list has numerous duplicates of the same entities due to:
* Use of **first/last name only** vs **first & last name references** (*Gatsby, Jordan* vs *Jay Gatsby, Jordan Baker*)
* **Possessives** (*Daisy vs Daisy's*)

#### Notable observation about SpaCy:
* SpaCy incorrectly tags possessives as `PERSON` (rather than as `PRON` (how it tags words like *my, your, yours, his, her, hers, *etc.).

### (Approach 2) Normalization of SpaCy's NER Output to account for false distinction observed

Given the issues described above, normalization of the NER's from SpaCy is necessary.

Thus, below, I do the following normalizations:
* merge references to the same person whether by first or last name only
* merge references using 1st person sg pronouns to the narrator (Nick)
* merge possessives to the person who is the possessor

*Note on possessive normalization*:
* As mentioned above, linguistically, tagging a possessive pronoun (*Gatsby's*, *Daisy's*, *my*, etc.) as a `PERSON` is not correct. IMO, it should just be treated as a grammatical possessive, which is`PRON` in SpaCy (notably, in SpaCy, there is no further distinction of types of pronouns e.g. `*PRONPOSS*`). Despite this, in the case of this analysis, I am choosing to normalize possessives to the `PERSON` because we are looking for all references to/mentions of that person in the book. Thus, to this end I am choosing to normalize this way.

In [10]:

# Consolidate counts (for main characters):
# Normalize for different combinations of first, last, first + last names
# Normalize for possessives (& different apostrophe variants)
# Normalize for 1st person references to Narrator (Nick)
# {key=observed form: val=baseline ref form}
NAME_MAP = {
    "Jay Gatsby": "Gatsby",
    "Jay": "Gatsby",
    "Gatsby's": "Gatsby",
    "James Gatz": "Gatsby",  # Gatsby's real name
    "Gatz": "Gatsby",
    "Gatsby\u2019s": "Gatsby",  # curly apostrophe variant
    "Tom Buchanan":"Tom",
    "Tom Buchanan's":"Tom",
    "Tom's'":"Tom",
    "Tom\u2019s": "Tom",
    "Jordan Baker": "Jordan",
    "Jordan's'": "Jordan",
    "Jordan Baker's": "Jordan",
    "Daisy Buchanan": "Daisy",
    "Daisy Buchanan's": "Daisy",
    "Daisy's": "Daisy",
    "Daisy\u2019s": "Daisy",
    "Nick Carraway": "Nick",
    "Nick Carraway\u2019s": "Nick",
    "Nick\u2019s": "Nick",
    "Carraway": "Nick",
    "I": "Nick",
    "me": "Nick",
    "George Wilson": "Wilson",
    "George B. Wilson": "Wilson",
    "Cody": "Dan Cody",
    "Dan Cody's": "Dan Cody",
    "Dan Cody\u2019s": "Dan Cody",
    "Wolfshiem": "Meyer Wolfshiem",
    "Meyer": "Meyer Wolfshiem",
    "Myrtle Wilson's": "Myrtle Wilson",
    "Myrtle Wilson\u2019s": "Myrtle Wilson"
    }

# dict.get(key, default val)
def normalize_names(name: str) -> str:
  return NAME_MAP.get(name, name)


# Modify the entity column: df["column name"].map(function_name)
# Create a normalized copy — leave entities_labels unchanged for Approach 1
person_entities_normalized = person_entities.copy()
person_entities_normalized["entity"] = person_entities_normalized["entity"].map(normalize_names)
person_entities_normalized["entity"].value_counts().head(35)

,count
entity,
Gatsby,262
Tom,186
Daisy,179
Jordan,41
Wilson,40
Baker,28
Nick,18
Meyer Wolfshiem,17
McKee,15


In [11]:
# Comparison of original vs normalized person entity counts
approach1_counts = person_entities["entity"].value_counts()
approach2_counts = person_entities_normalized["entity"].value_counts()

comparison = pd.DataFrame({
    "before": approach1_counts,
    "after": approach2_counts
}).fillna(0).astype(int)

comparison["difference"] = comparison["after"] - comparison["before"]
comparison.sort_values("difference", ascending=False)[:10]

,before,after,difference
entity,,,
Tom,161,186,25
Gatsby,239,262,23
Meyer Wolfshiem,0,17,17
Daisy,166,179,13
Jordan,29,41,12
Dan Cody,6,15,9
Nick,11,18,7
Wilson,35,40,5
Myrtle Wilson,3,6,3


### Analysis Approach 2:

As shown in the table above, with this normalization, the counts of the following characters (`PERSON` entities) increased as follows:

 * Tom: +25
 * Gatsby: + 23
 * Meyer Wolfshiem: +17
 * Daisy: +13
 * Jordan: +12
 * Dan Cody: + 9
 * Nick: +7
 * Wilson (George): +5
 * Myrtle Wilson: +3

### Correlation between character mentions & prominence in novel:

The protagonist (Nick) occurs 7th most frequently, this is be cause he is the narrator. Despite this, the story is centered around Gatsby, his home, and social circle. Everyone the narrator encounters is through Gatsby in some way or another, so Gatsby is mentioned the most by far. This is evinced in the character count totals as Gatsby is mentioned 262 times throughout the novel, 76 more times than the second most mentioned character (Tom).

### NER using Transformers

In [12]:
from transformers import pipeline

# Load transformer NER pipeline
ner = pipeline("token-classification", model="Jean-Baptiste/roberta-large-ner-english", aggregation_strategy="first")

# --- chunk the text before running through the pipeline ---
def chunk_text(text, chunk_size=400):
    words = text.split()
    return [" ".join(words[i:i+chunk_size])
            for i in range(0, len(words), chunk_size)]

chunks = chunk_text(text)

# run NER on each chunk instead of the whole text at once ---
all_results = []
for chunk in chunks:
    all_results.extend(ner(chunk))

# Map transformer label names to match your original target labels
label_map = {
    "PER": "PERSON",
    "LOC": "GPE",
    "ORG": "ORG"
}

# Filter and build lists — mirrors your original loop exactly
target = ["PERSON", "GPE", "ORG"]
entities = []
labels = []

for ent in all_results:
    mapped_label = label_map.get(ent["entity_group"], ent["entity_group"])
    if mapped_label in target:
        entities.append(ent["word"])
        labels.append(mapped_label)

# Build DataFrame — identical structure to your original
entities_labels_transformers = pd.DataFrame({
    "entity": entities,
    "label": labels
})

entities_labels_transformers

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: Jean-Baptiste/roberta-large-ner-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:447: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


,entity,label
0,F. Scott Fitzgerald,PERSON
1,Zelda,PERSON
2,Thomas Parke d’Invilliers,PERSON
3,East,GPE
4,"Gatsby,",PERSON
...,...,...
1313,Sound.,GPE
1314,Gatsby’s,PERSON
1315,Gatsby’s,PERSON
1316,Daisy’s,PERSON


### Postprocessing needed for Transformers NER output

Whereas spaCy processes text it first runs a tokenizer that splits the raw string into tokens before any NER happens, the transformer pipeline does the opposite.

RoBERTa uses subword tokenization (specifically Byte-Pair Encoding) which splits words into meaning-preserving subword pieces for the purpose of handling vocabulary. It is optimized for semantic understanding, not surface-form cleanup. So "Gatsby." might be tokenized as ["Gatsby", "."] internally, but when the aggregation step reassembles the entity span from the raw text, it grabs the original character sequence — punctuation and apostrophes included — which is why postprocessing to strip these artifacts is necessary for the transformer output but not for spaCy.

Thus, we need to do additional postprocessing before performing token value counts for the characters in the novel.

In [13]:
# Post processing cleaning punctuation
from itertools import combinations

def clean_entities(entity_text): # run each step on/update 'entity_text'
  # Remove trailing punctuation (. , ! ?)
  entity_text = entity_text.strip(".,!?;:")
  # Remove possessives — handle ALL common apostrophe variants
  entity_text = entity_text.replace("'s", "")   # ASCII          U+0027
  entity_text = entity_text.replace("\u2019s", "")  # curly right    U+2019
  entity_text = entity_text.replace("\u2018s", "")  # curly left     U+2018
  entity_text = entity_text.replace("\u02BCs", "")  # modifier letter U+02BC
  # Strip any remaining whitespace
  entity_text = entity_text.strip()
  return entity_text

In [14]:
# Apply function to transformers output

# Revise content of entities_labels_transformers["entity"], redefine
# df["col"].apply() cleaning to the "entity" column in DF entities_labels_transformers
# note no parameter arg necessary in apply(clean_entities) because its attached to df column and that's the parameter
entities_labels_transformers["entity"] = entities_labels_transformers["entity"].apply(clean_entities)
# Also, apply name normalization applied to SpaCy output
entities_labels_transformers["entity"] = entities_labels_transformers["entity"].apply(normalize_names)

#Extract only Characters (PERSON) NEs
person_entities_transformers = entities_labels_transformers.loc[entities_labels_transformers["label"] =="PERSON",["entity"]]

### Value Counts of the top 20 characters (Transformers)

In [24]:
person_entities_transformers["entity"].value_counts().head(30)

,count
entity,
Gatsby,214
Daisy,139
Tom,138
Wilson,57
Jordan,54
Miss Baker,21
Meyer Wolfshiem,21
Michaelis,19
Nick,17


### Comparison of the 3 NER Character count methods:

In [25]:
spacy_raw_counts = person_entities["entity"].value_counts()
spacy_normalized_counts = person_entities_normalized["entity"].value_counts()
transformer_counts = person_entities_transformers["entity"].value_counts()

# Series objects in value_counts() make the entity name the index, thus these can be aligned
comparison_df = pd.DataFrame({
    "spaCy (raw)": spacy_raw_counts,
    "spaCy (normalized)": spacy_normalized_counts,
    "Transformers": transformer_counts
}).fillna(0).astype(int)

comparison_df.sort_values("spaCy (normalized)", ascending=False).head(20)

,spaCy (raw),spaCy (normalized),Transformers
entity,,,
Gatsby,239,262,214
Tom,161,186,138
Daisy,166,179,139
Jordan,29,41,54
Wilson,35,40,57
Baker,28,28,2
Nick,11,18,17
Meyer Wolfshiem,0,17,21
Dan Cody,6,15,12


### Analysis (Extracting NE/Characters from Great Gatsby):

**Spacy is better than Transformers for the purpose of counting tokens, but it still needed additional normalization!**

While Transformers performed fairly well for some character counts, and in a few cases better than Approach 1 (raw SpaCy), in others, there were still significant problems, namely:

* Counts of the 3 most oft-mentioned characters (Gatsby, Tom & Daisy) are significantly lower in the transformers version. Of note though, some (however not all), of the other characters, mentioned less than 100 times, are found in similar counts.

* Some counts in the Transformers output were chronically low, e.g. "Biloxi" who is a person (nicknamed that due to the fact that they are from Biloxi Mississippi) was only identified as a PERSON once, whereas in the two SpaCy models, there were seven mention counts. Additionally there were egregious disparities in characters like 'Baker', mentioned 28 times as per the SpaCy models and only 2 in the Transformers.

* There are numerous cases where despite the application of `clean_entities` (which sucessfully normalized possessives from the original Transformers NER output), which should've removed all punctuation, we still find the following in the output punctuation: *Nick,”*, *Tom,”*,*Myrtle?”*,*God!”*,*Daisy,”*

Noteworthy is how Transformers processes text:

**Chunk boundary problem**:
Due to the maximum input limit of RoBERTa, Transformers has to process the entire novel in subset under the maximum amount (which is 512 tokens). So when splitting the text into 400-word chunks, some mentions of "Gatsby" fall at the edge of a chunk — mid-sentence or mid-context. Thus, given that it classifies based on surrounding context and not pretrained tags, the transformer may lose the surrounding context it needs to confidently identify the entity, thus it can miss an entity or labels it differently.

When you split the novel into 400-word chunks, the following happens at a boundary:

* ...end of chunk 4:    _"He looked over at"_
* start of chunk 5:     _"Gatsby who was standing..."_

When the model processes chunk 5, Gatsby appears at the very start with almost no preceding context — just a sentence fragment. The self-attention mechanism has very little to attend to on the left side.

In contrast, SpaCy processes the whole document as a continuous stream of sentences.

**Context dependency**:
Transformers RoBERTa model uses context in parsing text so that it can be sensitive to varying senses of a given word. ie. _"Each token in the input sequence looks at ALL other tokens in the sequence to gather relevant information"_

Thus, the Transformer approach is more context-sensitive than spaCy — which may be an advantage in some applications, however it has disadvantages too. It can miss some cases that seem fairly obvious to humans and which a model like SpaCy's NER, which was trained directly on labeled entity spans would easily catch. So in sentences where "Gatsby" may appear in an unusual syntactic position, the model might not be confident enough to tag it. spaCy's rule-based components are more consistent about proper nouns regardless of context.

**Capitalization and surface form variation**:
Across a novel, the same character gets referred to as Gatsby, Jay Gatsby, Jay, Mr. Gatsby, old sport etc. Each of these is a separate recognition decision for both models, but the transformer may handle some of these forms less consistently across chunks. Normalization for each character was applied to both models so that every character would have a canonical form that all variants would be normalized to, however in the Transformers, this only moderately increased the totals which for the main characters (Gatsby, Tom, Daisy, Wilson, Jordan etc.) the counts were still significantly less than in the normalized Spacy output.

---
## Contextual Descriptors for Main Character (Gatsby)



Herein, I extract all adjectives that appear near mentions of "Gatsby" (or variants thereof defined as a set `GATSBY_NAMEMAP`). Capturing these surrounding adjectives  gives some insight into how the character is described and portrayed.


In [16]:
# Main character is Gatsby, defined as all variants thereof
GATSBY_NAMEMAP = {"Jay Gatsby", "Gatsby","Jay","Gatsby's","James Gatz", "Gatsby\u2019s"}

adjectives = []
sentences = list(doc.sents)
for sentence in sentences:

  if any(variant in sentence.text for variant in GATSBY_NAMEMAP):
    # gets list of indexes for entity in
    entity_indexes = [i for i, tok in enumerate(sentence) if tok.text in GATSBY_NAMEMAP]
    # for testing
    #print(f"{entity}") #\t {sentence}
    #print(f"{sentence} \n") #\t {sentence}

    for ent_ind in entity_indexes:
      # get slice of 5 indexes following entity
      next_five_indexes = sentence[ent_ind+1:ent_ind + 6]
      # get slice of 5 indexes previous entity
      prev_five_indexes = sentence[ent_ind-1:ent_ind - 6]
      # get .text, .pos_ in tuple of 5 tokens following entities
      next_five_tokens = [(token.text, token.pos_) for token in next_five_indexes]
      # get .text, .pos_ in tuple of 5 tokens previous entities
      prev_five_tokens = [(token.text, token.pos_) for token in prev_five_indexes]

      # for testing
      #print("5 following tokens:", next_five_tokens)
      #print("5 following pos:", next_five_pos)

      # append adjectives from next_five_tokens
      for token, pos in next_five_tokens:
        if pos == "ADJ":
          adjectives.append(token)

      # append adjectives from prev_five_tokens
      for token, pos in prev_five_tokens:
        if pos == "ADJ":
          adjectives.append(token)

#print(f"ADJECTIVES LIST: {adjectives}")
adjective_list = pd.DataFrame(
    {
        "adjectives": adjectives
    }
)


### Value Counts of the Adjectives

In [17]:
adjective_list["adjectives"].value_counts()[:20]

,count
adjectives,
little,3
long,3
enormous,2
majestic,2
gorgeous,2
florid,2
same,2
careful,2
foul,2


### Adjectival co-occurrences with main character (Gatsby):

The adjectives co-occurring with the main character (Gatsby) are not representative of him, nor should they be expected to be. Instead, they are adjectives describing the scenes, objects and other contextual information with which Gatsby interacts and/or in which he is located.

---

## N-Gram Analysis

In this section, I perform the following sets of value counts:
1. Bigrams from the ***original tokens***
2. Bigrams from lemmas ***w/ stopwords removed***
3. Trigrams from lemmas ***w/ stopwords removed***

All sets of value counts filter out the following:
- Punctuation, defined where a token's part of speech attribute, `.pos_`, equals the string `"PUNCT"`.
- Named entities, defined where a token's `.ent_iob_` attribute does not equal `"O"` (an abbreviation for "outside" entity).

### Bigram Value Counts from Original Text


In [18]:
rows = []

for sentence in doc.sents:
    # remove the PUNCT and named entities
    # note: this creates inaccurate N-grams!
  punct_filtered = [token for token in sentence if not token.is_punct and token.ent_iob_ == "O"]
  bigrams = [punct_filtered[i:i+2] for i in range(len(punct_filtered) -1)]
  for pair in bigrams:
    # (tok.text for tok in pair) converts to text
    joined_gram = " ".join(tok.text for tok in pair)
    rows.append(joined_gram)


bigrams_list = pd.DataFrame(
    {
        "bigrams": rows
    }
)

bigrams_list["bigrams"].value_counts()

,count
bigrams,
of the,208
in the,195
on the,133
and the,112
I was,111
...,...
the melodious,1
arrived always,1
with girls,1


### Bigram Value Counts from Preprocessed Text

This time the bigrams are created by filtering out stopwords and using lemmas instead of the original tokens.

In [19]:
lemma_rows = []

for sentence in doc.sents:
    # remove the PUNCT, named entities and stopwords
    # gets lemmas
    # note: this creates inaccurate N-grams!
  punct_filtered = [token.lemma_ for token in sentence if (not token.is_punct) and (token.ent_iob_ == "O") and (not token.is_stop)]
  bigrams = [punct_filtered[i:i+2] for i in range(len(punct_filtered) -1)]
  for pair in bigrams:
    joined_gram = " ".join(pair)
    lemma_rows.append(joined_gram)

#print(lemma_rows)
lemma_bigrams = pd.DataFrame({"bigrams": lemma_rows})

lemma_bigrams["bigrams"].value_counts()

,count
bigrams,
old sport,45
shake hand,13
young man,12
know say,10
good night,10
...,...
enter life,1
life know,1
know disapprove,1


### Comparing the effect on literary analysis of raw bigrams (*with stopwords*) vs preprocessed bigrams (*stopwords removed*):

The bigrams with the stopwords included is made up of mostly preposition and connector word pair (e.g. *in to, on the, in his*, etc..) and narrative phrases (e.g. *he said, he had, she said, I said*, etc.). Nothing is contentful or necessarily semantically unique to the story.

The main insights one could gain is what the narrative is that there is a first person narrative based on the first person phrases (e.g. *I said, I was, and I*...).

There are some potential insight from bigrams counts for phrases such as: (3rd sg masculine) *"he was", "he had", "he said", "that he", "in his", "and he"*, and (1st singular)* "I had", "I'd", "I do", "I've", "I'll", "I did"* each of these have a much higher count than similar phrases with the 3rd person singular female pronoun. This is a strong correlation with the fact that the protagonist is the narrator and the other primary character through which everything is set, is Gatsby, a male.

Without stopwords, though smaller in frequency, the bigrams show several phrases that are semantically insightful to the story (e.g. *"old sport"* is the most frequent which is famously one of Gatsby's favorite phrases, additionally even phrases like *"good night", "shake hand"* are contextually relevant to the story as there are many social events and parties at night.). A phrase like *"old sport"* is insightful as it's a phrase that is pretty distinct to the 1920's, thus, it is a good indicator of the time in which the story took place.

### Trigram Value Counts from Preprocessed Text

In this step trigrams are created. As in the previous step, stopwords are filtered out and lemmas are used instead of the original tokens.

In [20]:
lemma_rows = []

for sentence in doc.sents:
    # remove the PUNCT, named entities and stopwords
    # gets lemmas
    # note: this creates inaccurate N-grams!
  punct_filtered = [token.lemma_ for token in sentence if (not token.is_punct) and (token.ent_iob_ == "O") and (not token.is_stop)]
  # range(len(span) -2) so that i+2 (at end) goes right till end, not over
  trigrams = [punct_filtered[i:i+3] for i in range(len(punct_filtered) -2)]
  for triple in trigrams:
    joined_gram = " ".join(triple)
    lemma_rows.append(joined_gram)

#print(lemma_rows)
lemma_trigrams = pd.DataFrame({"trigrams": lemma_rows})

lemma_trigrams["trigrams"].value_counts()[:10]

,count
trigrams,
old sport say,6
look old sport,4
oh ga od,3
chin raise little,2
ah h h,2
fool fool God,2
meet bad driver,2
go bad bad,2
thing go bad,2


### No useful insights for Trigrams:

The trigrams don't reveal much beyond what is revealed in the bigrams with *"old sport"* (as that phrase occurs in the top two bigrams *"old sport say"* (6) and *"look old sport"* (4)). With the exception of *"night old sport*" (2), the rest of the top 10 trigrams (which occur >= 3 times) seem to have been affected by the removal of stopwords and/or lemmatization as they mostly seem nonsensical (*"oh ga od", "chin raise little", "ah h h", "fool fool God", "meet bad driver","go bad bad", "thing go bad"*)

---

## Character Interactions

### Approach 1: Interactions with narrator (Nick)

Herein, I identify how often the narrator (Nick) is mentioned with other characters in the same sentences (suggesting interactions or relationships).

In [21]:
NICK_NAMEMAP = {"Nick Carraway","Nick Carraway's","Nick Carraway\u2019s", "Nick", "Nick's","Nick\u2019s", "Carraway","Carraway\u2019s", "I", "me"}

co_occurring = []

for sentence in doc.sents:
  if any(variant in sentence.text for variant in NICK_NAMEMAP):
    # check the PERSON entities
    for ent in sentence.ents:
      if ent.label_ == "PERSON" and ent.text not in NICK_NAMEMAP:
        co_occurring.append(ent.text)


co_occurring_chars = pd.DataFrame({"character": co_occurring})
# Normalize names to remove possessives from co-occurring characters
co_occurring_chars["character"] = co_occurring_chars["character"].map(normalize_names)

# get value_counts()
co_occurring_chars["character"].value_counts()[:25]

,count
character,
Gatsby,181
Tom,104
Daisy,101
Jordan,27
Wilson,23
Baker,17
Meyer Wolfshiem,10
Dan Cody,9
McKee,9


### Approach 2: Interactions between all characters (Improved version)

Herein, I identify the counts of all unique character interactions in the same sentences.

Of note here:

 - All character pairs included
 - Narrator (1st sg) pronouns are normalized and merged with "Nick" (his name) in `NICK_TOKENS`
 - All other characters normalized as per NAME_MAP
 - Normalized name pairs to account for variance in references

 Note also: `NICK_TOKENS` is accessed as a set, as opposed to a string (as in previous version which operated on a string) which prevents substring match inflation (e.g. *where "I" is found as part of word in a given sentence*)




In [22]:

from itertools import combinations

co_occurrences =[]
# contractions like "I'd" tokenize as ["I", "'d"] so "I" already covers as it's already tokenized
NICK_TOKENS = {"I", "me"}

for sentence in doc.sents:
    total_persons = sum(1 for ent in sentence.ents if ent.label_ == "PERSON") + sum(1 for tok in sentence if tok.text in NICK_TOKENS)
    if total_persons > 1:
        unique_people = set([ent.text for ent in sentence.ents if ent.label_ == "PERSON"])
        # Replacing "I", "me" with "Nick" since that is the character the pronouns are representing
        if any(token.text in NICK_TOKENS for token in sentence):
            unique_people.add("Nick") # normalize for 1st sg
        # append (unique) pair to list
        co_occurrences.extend(combinations(sorted(unique_people), 2))

# Consolidate counts (for main characters):
# Normalize for different combinations of first, last, first + last names
NAME_MAP = {
    "Jay Gatsby": "Gatsby",
    "Jay": "Gatsby",
    "Gatsby's": "Gatsby",
    "James Gatz": "Gatsby",  # Gatsby's real name
    "Gatz": "Gatsby",
    "Gatsby\u2019s": "Gatsby",  # curly apostrophe variant
    "Tom Buchanan":"Tom",
    "Tom Buchanan's":"Tom",
    "Tom's'":"Tom",
    "Tom\u2019s": "Tom",
    "Jordan Baker": "Jordan",
    "Baker": "Jordan",
    "Jordan's'": "Jordan",
    "Jordan Baker's": "Jordan",
    "Daisy Buchanan": "Daisy",
    "Daisy Buchanan's": "Daisy",
    "Daisy's": "Daisy",
    "Daisy\u2019s": "Daisy",
    "Nick Carraway": "Nick",
    "Nick Carraway\u2019s": "Nick",
    "Nick\u2019s": "Nick",
    "Carraway": "Nick",
    "George Wilson": "Wilson",
    "George B. Wilson": "Wilson",
    "Cody": "Dan Cody",
    "Dan Cody's": "Dan Cody",
    "Dan Cody\u2019s": "Dan Cody",
    "Wolfshiem": "Meyer Wolfshiem",
    "Meyer": "Meyer Wolfshiem",
    "Myrtle Wilson's": "Myrtle Wilson",
    "Myrtle Wilson\u2019s": "Myrtle Wilson"
    }

# normalize name variants of ents and add pairs to a tuple
co_occurrences = [
    tuple(NAME_MAP.get(x,x) for x in pair)
    for pair in co_occurrences
    ]

# add to Pandas DF and display value_counts()
co_occurring_chars = pd.DataFrame({"character pairs": co_occurrences})
co_occurring_chars["character pairs"].value_counts()[:20]


,count
character pairs,
"(Gatsby, Nick)",113
"(Daisy, Nick)",69
"(Nick, Tom)",61
"(Jordan, Nick)",28
"(Daisy, Tom)",25
"(Daisy, Gatsby)",20
"(Gatsby, Tom)",16
"(Nick, Wilson)",8
"(Jordan, Tom)",7


#### Checking for inflation of "I" instances in Approach 1 and 2

In [23]:
# Test to check if using "I" as standalone match (Approach 1) may be inflated

# how many sentences does Approach 1 think contain Nick?
nick_sentences_a1 = [s for s in doc.sents if any(variant in s.text for variant in NICK_NAMEMAP)]
print(len(nick_sentences_a1))

# how many sentences does Version 2 think contain Nick?
nick_sentences_v2 = [s for s in doc.sents if any(tok.text in NICK_TOKENS for tok in s)]
print(len(nick_sentences_v2))

1974
1262


### Character co-occurrence analysis:

Based on the results, the characters that have the strongest correlation is Nick (the narrator) and Gatsby (*113 occurrences as per Approach 2 results*). As mentioned above, Nick's relationship with Gatsby (which begins as an acquaintance and evolves over the course of the story) is the window through which we are presented the vast majority of the entire story.

Additionally, the Daisy/Tom (husband and wife) and Daisy/Gatsby pairs at 25 and 20 occurrences each reveal a relationship triangle that exists in this social circle. Note however that as the narrator is a character interacting with everyone, there are actually more occurrences of Nick/Daisy (69) and Nick/Tom (61) than the others, these totals are likely a result of the narrative choice in the novel, rather than an indication that the importance of the relationship of Tom, Daisy and the narrator.

#### Notes on co-occurrence analysis process:

After trying to just extract co-occurrences just having tokenized the text, there were numerous inefficiencies. In order to accurately account for all true character co-occurrences, I needed to perform normalization. Specifically, I needed to **normalize for variant references to all characters (e.g. Gatsby is also referred to as: "Jay Gatsby", "James Gatz", "Gatz", and possessive "Gatsby's"). Additionally, it was necessary to account for the fact that the narrative is in the 1st person, and that the narrator is named Nick**

#### * On mismatch between counts of "Gatsby" co-occurring with "Nick" in 'Character co-occurrences' in Approach 1 and Approach 2:

The counts of co-occurrences between the narrator and Gatsby differ significantly in these two approaches. Whereas Approach 1 showed 181 co-occurrences, Approach 2 showed 113 co-occurrences. The more accurate version is most likely Approach 2 due to the way that the pronoun "I" is operated on in the mapping of these different approaches. The distinction is as follows:

*   Approach 1: variant in `sentence.text — in` operates on a *string*, so it does substring matching
*   Approach 2: `tok.text in NICK_TOKENS — in` operates on a *set*, so it does exact membership testing


To check on this, in the cell '***Checking for inflation of "I" instances in Approach 1 and 2***' above, I performed a general test to see the variance in the counts of all sentences with terms from the `NICK_NAMEMAP` (used in Approach 1) and `NICK_TOKENS` (from Approach 2). The results show an extreme disparity, with the matching method used in Approach 1 having 1974 matches, and that of Approach 2 having 1262 matches.

The reason for this error is that by matching string-level substrings to find mentions of Nick in a sentence (namely by the variant reference "I"), any instance where a capital "I" is found in the sentence is counted as a match for Nick, including as part of another word (likely sentence initial words starting with "i"). The revised Approach 2 using token-level matching, only retrieves the first person pronouns "I" or "me" thus is clearly the more accurate result.


